In [1]:
%run base.ipynb

### Import original Data 

In [2]:
df = get_original_data()

In [6]:
df.head()

,Category,Subcategory,Country,Launched,Deadline,Goal,Pledged,Backers,State,Duration
0,Fashion,Fashion,United_States,2009-04-21 21:02:48,2009-05-31,1000,625,30,Failed,39
1,Film__Video,Shorts,United_States,2009-04-23 00:07:53,2009-07-20,80000,22,3,Failed,87
2,Art,Illustration,United_States,2009-04-24 21:52:03,2009-05-03,20,35,3,Successful,8
3,Technology,Software,United_States,2009-04-25 17:36:21,2009-07-14,99,145,25,Successful,79
4,Fashion,Fashion,United_States,2009-04-27 14:10:39,2009-05-26,1900,387,10,Failed,28


### Clean categorical strings from spaces and troublesome special characters

In [7]:
import re

# Define a function to clean up string values
def clean_string(s):
    # Remove leading/trailing spaces
    s = s.strip()
    # Replace all spaces with underscores (or remove them if desired)
    s = re.sub(r'\s+', '_', s)  # Replace spaces with underscores
    # Remove problematic special characters
    s = re.sub(r'[^\w\s]', '', s)
    return s

# Apply the function to all columns in the DataFrame
df = df.applymap(lambda x: clean_string(x) if isinstance(x, str) else x)


In [8]:
df.head()

,Category,Subcategory,Country,Launched,Deadline,Goal,Pledged,Backers,State,Duration
0,Fashion,Fashion,United_States,2009-04-21 21:02:48,2009-05-31,1000,625,30,Failed,39
1,Film__Video,Shorts,United_States,2009-04-23 00:07:53,2009-07-20,80000,22,3,Failed,87
2,Art,Illustration,United_States,2009-04-24 21:52:03,2009-05-03,20,35,3,Successful,8
3,Technology,Software,United_States,2009-04-25 17:36:21,2009-07-14,99,145,25,Successful,79
4,Fashion,Fashion,United_States,2009-04-27 14:10:39,2009-05-26,1900,387,10,Failed,28


In [9]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 331462 entries, 0 to 374605
Data columns (total 10 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   Category     331462 non-null  object        
 1   Subcategory  331462 non-null  object        
 2   Country      331462 non-null  object        
 3   Launched     331462 non-null  datetime64[ns]
 4   Deadline     331462 non-null  datetime64[ns]
 5   Goal         331462 non-null  int64         
 6   Pledged      331462 non-null  int64         
 7   Backers      331462 non-null  int64         
 8   State        331462 non-null  object        
 9   Duration     331462 non-null  int64         
dtypes: datetime64[ns](2), int64(4), object(4)
memory usage: 27.8+ MB


### Encode categorical features as dummy (binary) variables

In [10]:
df = pd.get_dummies(df, drop_first=True, columns=df.select_dtypes(include=['object']).columns)


In [11]:
df.head()

,Launched,Deadline,Goal,Pledged,Backers,Duration,Category_Comics,Category_Crafts,Category_Dance,Category_Design,...,Country_Netherlands,Country_New_Zealand,Country_Norway,Country_Singapore,Country_Spain,Country_Sweden,Country_Switzerland,Country_United_Kingdom,Country_United_States,State_Successful
0,2009-04-21 21:02:48,2009-05-31,1000,625,30,39,False,False,False,False,...,False,False,False,False,False,False,False,False,True,False
1,2009-04-23 00:07:53,2009-07-20,80000,22,3,87,False,False,False,False,...,False,False,False,False,False,False,False,False,True,False
2,2009-04-24 21:52:03,2009-05-03,20,35,3,8,False,False,False,False,...,False,False,False,False,False,False,False,False,True,True
3,2009-04-25 17:36:21,2009-07-14,99,145,25,79,False,False,False,False,...,False,False,False,False,False,False,False,False,True,True
4,2009-04-27 14:10:39,2009-05-26,1900,387,10,28,False,False,False,False,...,False,False,False,False,False,False,False,False,True,False


In [12]:
df.shape

(331462, 200)

### Define target & features

In [13]:
import pandas as pd
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, make_scorer

# Drop or convert datetime columns
X = df.drop(columns=['State_Successful', 'Launched', 'Deadline', 'Pledged']) # Drop datetime columns + Pledged (data leakage)
y = df['State_Successful']  # Target column


In [14]:
# Extract features from datetime columns (here only the year to avoid too many features)
X['year_launched'] = df['Launched'].dt.year
# X['month_launched'] = df['Launched'].dt.month
# X['day_launched'] = df['Launched'].dt.day

X['year_deadline'] = df['Deadline'].dt.year
# X['year_deadline'] = df['Deadline'].dt.month
# X['year_deadline'] = df['Deadline'].dt.day

### Split the data into training and test sets

In [15]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

### Scale the data

In [16]:
# Scale numerical data using StandarScaler:

from sklearn.preprocessing import StandardScaler

# Identify numerical columns (excluding date-time and timedelta types)
numerical_cols = X.select_dtypes(include=['number']).columns.tolist()

# Initialize StandardScaler
scaler = StandardScaler()

# Fit and transform only the numerical columns to the training data
X_train[numerical_cols] = scaler.fit_transform(X_train[numerical_cols])

# Use the same scaler to transform the test data
X_test[numerical_cols] = scaler.transform(X_test[numerical_cols])

### Resample the data for imbalanced target classes
Since the resampling approach using SMOTE from the imbalanced-learn library completely crashed the local machine (MacBook Air M2 2022, 8GB, Sonoma 14.5) and the approach of randomly under-sampling the majority target class, while being computationally cheaper, will risk us losing meaningful information, there doesn't seem to be a clear optimal solution yet. By using the simplest and computationally cheapest form of resampling imbalanced target classes, namely by randomly under-sampling the majority class of the target variable, we apply the most (and only) feasible resampling method to handle imbalanced target classes within our limited computational means:

In [17]:
# Apply random undersampling to the training set
from imblearn.under_sampling import RandomUnderSampler
undersampler = RandomUnderSampler(random_state=42)
X_train_resampled, y_train_resampled = undersampler.fit_resample(X_train, y_train)

### Prepare the cross-validation strategy
- Stratified K-fold for better representation of classes
- Saga solver for faster convergence times
- Small number of max. iteration for computational limitations
- logistic regression with weights for additional class balance
- Ridge and Lasso regularization for feature selection insights
- Smaller regularization options for more aggressive feature selection due to large dimensionality of dummy encoded data

In [18]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.metrics import make_scorer, f1_score

# Define the StratifiedKFold cross-validator
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Define F1 scorer
f1_scorer = make_scorer(f1_score, average='weighted')

# Define the logistic regression model with class_weight='balanced'
logistic_regression = LogisticRegression(solver='saga', max_iter=100, class_weight='balanced')

# Define the parameter grid for Logistic Regression
param_grid_lr = {
    'penalty': ['l1', 'l2'],
    'C': [0.01, 0.1, 1, 10]
    }

# GridSearchCV for Logistic Regression
grid_search_lr = GridSearchCV(logistic_regression, param_grid_lr, cv=skf, scoring=f1_scorer, n_jobs=-1)

### Apply stratified k-fold cross validation gridsearch
Fitted to the resampled training data.

In [20]:
grid_search_lr.fit(X_train_resampled, y_train_resampled)

/Users/laylanyrabia/neuefische/kickstarter/project_kickstarter/.venv/lib/python3.11/site-packages/sklearn/linear_model/_sag.py:350: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/Users/laylanyrabia/neuefische/kickstarter/project_kickstarter/.venv/lib/python3.11/site-packages/sklearn/linear_model/_sag.py:350: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/Users/laylanyrabia/neuefische/kickstarter/project_kickstarter/.venv/lib/python3.11/site-packages/sklearn/linear_model/_sag.py:350: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/Users/laylanyrabia/neuefische/kickstarter/project_kickstarter/.venv/lib/python3.11/site-packages/sklearn/linear_model/_sag.py:350: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/Users/laylanyrabia/neuefische/kickstarter/project_kickstarter/.venv

GridSearchCV(cv=StratifiedKFold(n_splits=5, random_state=42, shuffle=True),
             estimator=LogisticRegression(class_weight='balanced',
                                          solver='saga'),
             n_jobs=-1,
             param_grid={'C': [0.01, 0.1, 1, 10], 'penalty': ['l1', 'l2']},
             scoring=make_scorer(f1_score, average=weighted))

In [21]:
# Best Logistic Regression Model
best_lr = grid_search_lr.best_estimator_

# Output the best model and its corresponding hyperparameters
print("Best Logistic Regression Model:", best_lr)
print("Logistic Regression F1 Score:", f1_score(y_test, best_lr.predict(X_test), average='weighted'))

# Output the best hyperparameters

coefficients = best_lr.coef_.flatten()
feature_names = X_test.columns
coeff_df = pd.DataFrame({
    'Feature': feature_names,
    'Coefficient': coefficients
})
selected_features = coeff_df[coeff_df['Coefficient'] != 0]
selected_features = selected_features.reindex(selected_features['Coefficient'].abs().sort_values(ascending=False).index)
print(selected_features)

# Best Logistic Regression Model: LogisticRegression(C=0.01, class_weight='balanced', penalty='l1', solver='saga')
# Logistic Regression F1 Score: 0.7975915850439066
# 61 features selected

Best Logistic Regression Model: LogisticRegression(C=0.01, class_weight='balanced', penalty='l1', solver='saga')
Logistic Regression F1 Score: 0.7975915850439066
                      Feature   Coefficient
1                     Backers  1.015463e+01
0                        Goal -2.289651e+00
84         Subcategory_HipHop -9.202204e-01
16           Category_Theater  8.521464e-01
164   Subcategory_Video_Games -7.783138e-01
..                        ...           ...
56          Subcategory_Drama  1.381033e-02
55    Subcategory_Documentary -1.449379e-03
115  Subcategory_Performances  9.369830e-04
74           Subcategory_Food  3.724464e-08
106         Subcategory_Music -3.476722e-09

[61 rows x 2 columns]


### Feature Pre-Selection: 
Since the categorical features include a large number of unique values and the data set is of a 6-figure order, we might need to perform a more selective lasso regression prior to the modelling for this to be computationally feasible. This also means decreasing the regularization constant (C):

In [22]:
from sklearn.linear_model import LogisticRegression
from sklearn.feature_selection import SelectFromModel

# Step 1: Define a more selective Logistic Regression model by decreasing C
# Smaller C values increase regularization strength
logreg = LogisticRegression(penalty="l1", solver="saga", max_iter=100, C=0.001)

# Step 2: Fit the feature selector model with the more selective logistic regression
selector = SelectFromModel(logreg, threshold="mean")
selector.fit(X_train_resampled, y_train_resampled)

# Step 3: Apply the selector to reduce dimensionality
X_train_reduced = selector.transform(X_train_resampled)
X_test_reduced = selector.transform(X_test)

# Step 4: Retrieve the indices and names of selected features
selected_feature_indices = selector.get_support(indices=True)
selected_feature_names = np.array(feature_names)[selected_feature_indices]

print("Number of selected features:", len(selected_feature_indices))
print("Selected feature indices:", selected_feature_indices)
print("Selected feature names:", selected_feature_names)
print("Shape of the reduced feature set:", X_train_reduced.shape)

# C=0.0001, 0.001 (feature selection with L1 regularization reached saturation)
# Number of selected features: 10
# Selected feature indices: [  0   1   2   9  12  15  16 145 164 197]
# Selected feature names: ['Goal' 'Backers' 'Duration' 'Category_Food' 'Category_Music'
# 'Category_Technology' 'Category_Theater' 'Subcategory_Shorts'
# 'Subcategory_Video_Games' 'year_deadline']
# Shape of the reduced feature set: (214162, 10)


Number of selected features: 10
Selected feature indices: [  0   1   2   9  12  15  16 145 164 197]
Selected feature names: ['Goal' 'Backers' 'Duration' 'Category_Food' 'Category_Music'
 'Category_Technology' 'Category_Theater' 'Subcategory_Shorts'
 'Subcategory_Video_Games' 'year_deadline']
Shape of the reduced feature set: (214162, 10)


/Users/laylanyrabia/neuefische/kickstarter/project_kickstarter/.venv/lib/python3.11/site-packages/sklearn/linear_model/_sag.py:350: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


In [23]:
# Define the logistic regression model with class_weight='balanced'
logistic_regression = LogisticRegression(solver='saga', max_iter=1000, class_weight='balanced')

# Define the parameter grid for Logistic Regression
param_grid_lr = {
    'penalty': ['l2', 'None', 'elasticnet'],
    'C': [0.01, 0.1, 1, 10, 100]
    }
grid_search_lr = GridSearchCV(logistic_regression, param_grid_lr, cv=skf, scoring=f1_scorer, n_jobs=-1)
grid_search_lr.fit(X_test_reduced, y_test)

/Users/laylanyrabia/neuefische/kickstarter/project_kickstarter/.venv/lib/python3.11/site-packages/sklearn/linear_model/_sag.py:350: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/Users/laylanyrabia/neuefische/kickstarter/project_kickstarter/.venv/lib/python3.11/site-packages/sklearn/linear_model/_sag.py:350: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/Users/laylanyrabia/neuefische/kickstarter/project_kickstarter/.venv/lib/python3.11/site-packages/sklearn/linear_model/_sag.py:350: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/Users/laylanyrabia/neuefische/kickstarter/project_kickstarter/.venv/lib/python3.11/site-packages/sklearn/linear_model/_sag.py:350: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/Users/laylanyrabia/neuefische/kickstarter/project_kickstarter/.venv

GridSearchCV(cv=StratifiedKFold(n_splits=5, random_state=42, shuffle=True),
             estimator=LogisticRegression(class_weight='balanced',
                                          max_iter=1000, solver='saga'),
             n_jobs=-1,
             param_grid={'C': [0.01, 0.1, 1, 10, 100],
                         'penalty': ['l2', 'None', 'elasticnet']},
             scoring=make_scorer(f1_score, average=weighted))

In [24]:
# Best Logistic Regression Model
best_lr = grid_search_lr.best_estimator_

# Output the best model and its corresponding hyperparameters
print("Best Logistic Regression Model:", best_lr)
print("Logistic Regression F1 Score:", f1_score(y_test, best_lr.predict(X_test_reduced), average='weighted'))

# BBest Logistic Regression Model: LogisticRegression(C=100, class_weight='balanced', max_iter=1000, solver='saga')
# Logistic Regression F1 Score: 0.856157477908096

Best Logistic Regression Model: LogisticRegression(C=100, class_weight='balanced', max_iter=1000, solver='saga')
Logistic Regression F1 Score: 0.8561574779080967
